In [1]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares

def ode_system(t, y, params):
    T, E, N = y
    rT, KT, kE, kN, sE, aE, dE, sN, aN, dN = params

    dTdt = rT*T*(1 - T/KT) - kE*E*T - kN*N*T
    dEdt = sE + aE*T - dE*E
    dNdt = sN + aN*T - dN*N

    return [dTdt, dEdt, dNdt]

def simulate(params, y0, t_eval):
    sol = solve_ivp(
        lambda t, y: ode_system(t, y, params),
        [t_eval[0], t_eval[-1]],
        y0,
        t_eval=t_eval,
        method="RK45"
    )
    return sol.y  # (3, len(t_eval))

def loss_function(params, y0, t_eval, data_E, data_N):
    T, E, N = simulate(params, y0, t_eval)

    res_E = E - data_E
    res_N = N - data_N

    return np.concatenate([res_E, res_N])

# timepoints
t_eval = np.array([3, 7, 14])  # replace 14 with your "end"

# insert your real densities here
data_E = np.array([0.8, 1.2, 1.5])  # CD8 density
data_N = np.array([0.5, 0.4, 0.3])  # NK density

# initial conditions (T0, E0, N0)
y0 = [1.0, 0.2, 0.2]

# initial parameter guess
p0 = np.array([0.3, 10.0, 0.01, 0.01, 0.05, 0.02, 0.1, 0.05, 0.02, 0.1])

result = least_squares(
    loss_function,
    p0,
    bounds=(0, np.inf),
    args=(y0, t_eval, data_E, data_N)
)

fitted_params = result.x
print("Fitted parameters:")
print(fitted_params)


Fitted parameters:
[ 0.57969343 10.10520543  0.8324259   0.76571088  0.2933628   0.15654412
  0.18387169  0.07350673  0.10415622  0.26733066]
